# SQL 전처리(pandas) 과제 2 - 이창현 (7597)
## 복습 키워드: pd.cut / pd.qcut / pivot / crosstab / 결측치 / MCAR

---
## 1. pd.cut — 직접 구간 지정 분할

### 개념
- 연속형 데이터를 **직접 지정한 구간(bins)** 으로 나눔
- 구간 경계값을 사용자가 직접 설정
- SQL의 CASE WHEN 구간 분류와 동일한 개념
- 예) 0~60=F, 60~80=C, 80~100=A 처럼 고정 구간 분류

In [ ]:
import pandas as pd
import numpy as np

# 예시 데이터
df = pd.DataFrame({
    'name':  ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Frank'],
    'age':   [22, 35, 48, 61, 74, 55],
    'score': [45, 72, 85, 91, 63, 78]
})

# ── pd.cut: 직접 구간 지정 ──
# SQL의 CASE WHEN age <= 30 THEN '20대이하' ... 과 동일
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 30, 50, 70, 100],                           # 구간 경계값 직접 지정
    labels=['20대이하', '30-40대', '50-60대', '70대이상'],  # 레이블
    right=True                                            # 오른쪽 경계 포함 (기본값)
)

print("=== pd.cut 결과 ===")
print(df[['name', 'age', 'age_group']])
print()

# 점수 등급 분류
df['grade'] = pd.cut(df['score'], bins=[0, 60, 80, 100], labels=['F', 'C', 'A'])
print("=== 점수 등급 분류 ===")
print(df[['name', 'score', 'grade']])
print()

# 구간별 빈도수
print("=== 나이 구간별 빈도수 ===")
print(df['age_group'].value_counts().sort_index())

---
## 2. pd.qcut — 분위수 기반 자동 분할

### 개념
- 연속형 데이터를 **분위수(quantile) 기준** 으로 자동 분할
- 각 구간의 **데이터 개수가 균등** 하도록 분할
- 구간 경계값은 데이터에 따라 자동 결정

### pd.cut vs pd.qcut 핵심 차이
| | pd.cut | pd.qcut |
|---|---|---|
| 구간 기준 | 값의 범위 (직접 지정) | 데이터 개수 (균등 분할) |
| 각 구간 크기 | 동일한 범위 | 동일한 개수 |
| 사용 시점 | 기준값이 명확할 때 | 균등 분포가 필요할 때 |

In [ ]:
# pd.qcut: 분위수 기반 자동 분할
df2 = pd.DataFrame({
    'name':  [f'고객{i}' for i in range(1, 11)],
    'sales': [150, 300, 450, 200, 800, 1200, 550, 900, 100, 650]
})

# 4분위로 자동 분할 (각 구간에 동일한 개수 배치)
df2['sales_quartile'] = pd.qcut(
    df2['sales'],
    q=4,
    labels=['하위25%', '25~50%', '50~75%', '상위25%']
)

print("=== pd.qcut 결과 (4분위) ===")
print(df2.sort_values('sales').reset_index(drop=True))
print()

# cut vs qcut 비교
df2['cut_group']  = pd.cut( df2['sales'], bins=4, labels=['1구간','2구간','3구간','4구간'])
df2['qcut_group'] = pd.qcut(df2['sales'], q=4,    labels=['1구간','2구간','3구간','4구간'])

print("=== cut vs qcut 구간별 빈도 비교 ===")
print("pd.cut  구간별 빈도:", df2['cut_group'].value_counts().sort_index().to_dict())
print("pd.qcut 구간별 빈도:", df2['qcut_group'].value_counts().sort_index().to_dict())
print()
print("→ qcut은 각 구간에 균등한 개수, cut은 균등한 범위")

---
## 3. pivot — 행을 열로 변환 (피봇팅)

### 개념
- 데이터를 **재구조화** 하여 행(Row)을 열(Column)로 변환
- SQL의 `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` 피봇팅과 동일
- `index` = 행 기준, `columns` = 열로 펼칠 기준, `values` = 값

In [ ]:
# pivot 예시: 월별 카테고리 지출 재구조화
data = {
    'month':    ['1월', '1월', '1월', '2월', '2월', '2월'],
    'category': ['식비', '교통', '쇼핑', '식비', '교통', '쇼핑'],
    'amount':   [300000, 50000, 150000, 250000, 60000, 200000]
}
df3 = pd.DataFrame(data)

print("=== 원본 데이터 (피봇팅 전) ===")
print(df3)
print()

# pivot으로 재구조화
pivot_result = df3.pivot(
    index='month',       # 행 기준
    columns='category',  # 열로 펼칠 기준
    values='amount'      # 채울 값
)

print("=== pivot 결과 (피봇팅 후) ===")
print(pivot_result)
print()
print("→ 세로로 쌓인 데이터를 가로로 펼쳐서 한눈에 비교 가능")

In [ ]:
# pivot_table: 중복값을 집계함수로 처리
data2 = {
    'gender': ['M','M','F','F','M','F'],
    'year':   [2023,2024,2023,2024,2023,2024],
    'sales':  [1000,1200,900,1100,800,950]
}
df4 = pd.DataFrame(data2)

pivot_tbl = df4.pivot_table(
    index='year',
    columns='gender',
    values='sales',
    aggfunc='sum'    # 집계함수 지정 (sum, mean, count 등)
)

print("=== pivot_table 결과 (성별 × 연도별 매출 합계) ===")
print(pivot_tbl)
print()
print("→ pivot은 중복값 오류 / pivot_table은 aggfunc으로 처리")

---
## 4. crosstab — 교차표 생성

### 개념
- 두 범주형 변수의 **빈도수 또는 비율** 을 교차표로 표현
- pivot_table의 특수한 형태 (기본값이 빈도수)
- 범주형 변수 간의 관계 파악에 유용

### pivot_table vs crosstab 차이
| | pivot_table | crosstab |
|---|---|---|
| 기본 집계 | 평균(mean) | 빈도수(count) |
| 입력 방식 | DataFrame 컬럼명 | Series 또는 배열 |
| 주요 용도 | 다양한 집계 | 빈도수 / 비율 분석 |

In [ ]:
# crosstab 예시
data3 = {
    'gender':    ['M','F','M','F','M','F','M','F'],
    'age_group': ['20대','20대','30대','30대','20대','30대','20대','30대'],
    'purchased': ['Y','N','Y','Y','N','Y','Y','N']
}
df5 = pd.DataFrame(data3)

# 기본 crosstab (빈도수)
ct_basic = pd.crosstab(df5['gender'], df5['age_group'])
print("=== crosstab 기본 (빈도수) ===")
print(ct_basic)
print()

# 비율로 변환 (normalize)
ct_ratio = pd.crosstab(df5['gender'], df5['age_group'], normalize='all')
print("=== crosstab 비율 (normalize='all') ===")
print(ct_ratio.round(2))
print()

# margins: 행/열 합계 추가
ct_margin = pd.crosstab(df5['gender'], df5['age_group'], margins=True)
print("=== crosstab 합계 포함 (margins=True) ===")
print(ct_margin)

---
## 5. 결측치 (Missing Value)

### 개념
- 데이터에 값이 없는 상태: `NaN`, `None`, `NULL`
- 분석 전 반드시 처리해야 하는 핵심 전처리 단계
- **처리 방법**: 제거(dropna) / 대체(fillna) / 예측(모델 활용)

### 결측치 발생 패턴 3가지
| 종류 | 설명 | 처리 방향 |
|---|---|---|
| **MCAR** | 완전 무작위 결측 | 제거해도 편향 없음 ✅ |
| **MAR** | 다른 관측 변수와 관련된 결측 | 조건부 대체 |
| **MNAR** | 결측값 자체와 관련된 결측 | 신중한 처리 필요 ⚠️ |

In [ ]:
# 결측치 확인 및 처리
df6 = pd.DataFrame({
    'name':   ['Alice','Bob','Charlie','David','Eve'],
    'age':    [25, np.nan, 35, 28, np.nan],
    'salary': [3000, 4000, np.nan, 3500, 4500],
    'dept':   ['영업','개발','영업', None,'개발']
})

print("=== 원본 데이터 ===")
print(df6)
print()

# 결측치 확인
print("=== 결측치 개수 ===")
print(df6.isnull().sum())
print()

print("=== 결측치 비율 ===")
print((df6.isnull().sum() / len(df6) * 100).round(1).astype(str) + '%')
print()

# 결측치 포함 행
print("=== 결측치 포함 행 ===")
print(df6[df6.isnull().any(axis=1)])

In [ ]:
# 결측치 처리 방법 비교

# 1. 제거 (dropna)
print("=== 1. 결측치 행 제거 (dropna) ===")
print(df6.dropna())
print()

# 2. 평균/중앙값으로 대체 (fillna)
df_fill = df6.copy()
df_fill['age']    = df_fill['age'].fillna(df_fill['age'].mean())       # 평균 대체
df_fill['salary'] = df_fill['salary'].fillna(df_fill['salary'].median()) # 중앙값 대체
df_fill['dept']   = df_fill['dept'].fillna('미지정')                    # 특정값 대체

print("=== 2. 결측치 대체 (fillna) ===")
print(df_fill)
print()

# 3. 앞/뒤 값으로 채우기
df_ffill = df6.copy()
df_ffill['age'] = df_ffill['age'].fillna(method='ffill')  # 앞 값으로 채우기

print("=== 3. 앞값으로 채우기 (ffill) ===")
print(df_ffill[['name','age']])

---
## 6. MCAR (Missing Completely At Random)

### 개념
- **완전 무작위 결측** — 결측 발생이 어떤 변수와도 무관
- 데이터 수집 과정의 순전한 우연으로 발생
- **예시**: 설문 작성 중 실수 누락 / 센서 랜덤 오류

### 왜 중요한가?
- MCAR이면 → 결측 행 제거해도 **분석 편향 없음** ✅
- MNAR이면 → 결측 행 제거하면 **분석 결과 왜곡** ⚠️

### 판별 방법
- 결측 여부와 다른 변수 간의 관계 확인
- 결측 그룹 / 비결측 그룹의 평균 비교
- Little's MCAR Test (통계적 검정)

In [ ]:
# MCAR vs MNAR 시뮬레이션 비교
np.random.seed(42)
n = 200
df_test = pd.DataFrame({
    'age':    np.random.randint(20, 60, n),
    'income': np.random.randint(3000, 8000, n)
})

# ── MCAR: 완전 무작위 결측 ──
mcar_idx = np.random.choice(df_test.index, size=40, replace=False)
df_test['income_MCAR'] = df_test['income'].copy()
df_test.loc[mcar_idx, 'income_MCAR'] = np.nan

# ── MNAR: 고소득자(7000 이상)가 소득 미기재하는 패턴 ──
mnar_idx = df_test[df_test['income'] >= 7000].index
df_test['income_MNAR'] = df_test['income'].copy()
df_test.loc[mnar_idx, 'income_MNAR'] = np.nan

print("=== MCAR vs MNAR 비교 ===")
print(f"원본 income 평균:       {df_test['income'].mean():.0f}원")
print(f"MCAR 제거 후 평균:      {df_test['income_MCAR'].mean():.0f}원  ← 원본과 유사 ✅")
print(f"MNAR 제거 후 평균:      {df_test['income_MNAR'].mean():.0f}원  ← 원본보다 낮음 ⚠️")
print()
print("→ MCAR: 제거해도 평균 유지 (편향 없음)")
print("→ MNAR: 제거하면 고소득자 제외 → 평균 낮아짐 (편향 발생)")

In [ ]:
# MCAR 검증: 결측 여부와 다른 변수의 관계 확인
df_test['missing_flag'] = df_test['income_MCAR'].isnull().astype(int)

print("=== MCAR 검증: 결측 그룹 vs 비결측 그룹 나이 비교 ===")
print(df_test.groupby('missing_flag')['age'].agg(['mean','std']).round(1))
print()
print("→ 두 그룹의 나이 평균/분산이 비슷하면 → MCAR 가능성 높음 ✅")
print("→ 두 그룹의 나이 평균이 크게 다르면 → MAR 또는 MNAR 의심 ⚠️")

---
## 핵심 정리

| 키워드 | 한 줄 요약 |
|---|---|
| **pd.cut** | 직접 구간 지정 → 고정된 범위로 분할 (SQL CASE WHEN과 유사) |
| **pd.qcut** | 분위수 기반 → 각 구간 데이터 개수 균등 분할 |
| **pivot** | 행을 열로 변환 → 데이터 재구조화 (피봇팅) |
| **crosstab** | 두 변수 교차표 → 빈도수/비율 분석 |
| **결측치** | NaN/None/NULL → dropna 제거 / fillna 대체 |
| **MCAR** | 완전 무작위 결측 → 제거해도 분석 편향 없음 ✅ |

### MCAR / MAR / MNAR 최종 비교
| 종류 | 결측 원인 | 예시 | 처리 |
|---|---|---|---|
| MCAR | 어떤 변수와도 무관 | 설문 실수 누락 | 제거 OK ✅ |
| MAR | 다른 관측 변수와 관련 | 고령자가 소득 미기재 | 조건부 대체 |
| MNAR | 결측값 자체와 관련 | 고소득자가 소득 미기재 | 신중한 처리 ⚠️ |